In [ ]:
import numpy as np  
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 
import plotly as py 
import plotly.graph_objs as go
import warnings 
warnings.filterwarnings('ignore')
import sheryanalysis as sh 

In [ ]:
df = pd.read_csv('online_shoppers_intention.csv')

In [ ]:
df.head()

In [ ]:
sh.analyze(df)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
# missing percent of data 
missing_percentage = df.isnull().sum()/df.shape[0]
missing_percentage

In [ ]:
df['Revenue'].value_counts()

In [ ]:
df.shape

In [ ]:
df.duplicated().sum()

In [ ]:
num_col = df.select_dtypes(include='number').columns
cat_col = df.select_dtypes(include='object').columns

In [ ]:
num_col

In [ ]:
cat_col

In [ ]:
sns.countplot(x='Revenue',data=df,palette='Set1')

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x = 'VisitorType',data=df,palette='Set1')
plt.title('Visitor type distribution')
plt.xticks(rotation=20)
plt.show()

In [ ]:
sns.countplot(x='VisitorType',hue='Revenue',data=df)

In [ ]:
sns.countplot(x='Month',data=df ,order=df['Month'].value_counts().index,palette='Set1')

In [ ]:
sns.countplot(x='Month',hue='Revenue',data=df,palette='dark')

In [ ]:
sns.histplot(df['Administrative_Duration'],bins=30,kde=True)

In [ ]:
sns.histplot(df['ProductRelated_Duration'],bins=40,kde=True)

In [ ]:
sns.boxplot(x='Revenue',y='BounceRates',data=df,palette='bright')

In [ ]:
sns.boxplot(x='Revenue',y='ExitRates',data=df,palette='dark')

In [ ]:
sns.boxplot(x='Revenue',y='PageValues',data=df,palette='rainbow')

In [ ]:
plt.figure(figsize=(14,10))
corr = df.corr(numeric_only=True)
sns.heatmap(corr,annot=True,fmt=".2f",cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
df = pd.get_dummies(df,drop_first=True)

In [ ]:
df.head().astype(int)

In [ ]:
X = df.drop('Revenue',axis=1)
y = df['Revenue']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report,confusion_matrix
from sklearn.model_selection import GridSearchCV

In [ ]:
le = LabelEncoder()
categorical_cols = df.select_dtypes(include='object').columns

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

# Convert Boolean Columns to Integer
df["Weekend"] = df["Weekend"].astype(int)
df["Revenue"] = df["Revenue"].astype(int)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## LOGISTIC REGRESSION

In [ ]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()
lr.fit(X_train,y_train)

In [ ]:
y_pred_lr = lr.predict(X_test)

In [ ]:
y_pred_lr

In [ ]:
accuracy = accuracy_score(y_test,y_pred_lr)
print(f"Accuracy:",accuracy*100,"%")

In [ ]:
params = {
    'C':[0.01,0.1,1,10,100],
    'penalty':['l2']
}

In [ ]:
grid = GridSearchCV(LogisticRegression(max_iter=1000),
    params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

## STANDARDSCALER

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

In [ ]:
grid.fit(
    X_train_scaled,
    y_train
)

In [ ]:
print(grid.best_params_)

In [ ]:
best_lr = grid.best_estimator_

y_pred = best_lr.predict(
    X_test_scaled
)

print(f"Accuracy",accuracy_score(y_test,y_pred)*100,"%"
)

## KNN HYPERPARAMETER TUNING 

In [ ]:
model_knn = KNeighborsClassifier(n_neighbors=5)
model_knn.fit(X_train_scaled,y_train)

In [ ]:
y_pred_knn = model_knn.predict(X_test_scaled)

In [ ]:
y_pred_knn

In [ ]:
accuracy = accuracy_score(y_test,y_pred_knn)
print(f"Accuracy:",accuracy*100,"%")

In [ ]:
params = {
    'n_neighbors':[3,5,7,9,11],
    'weights':['uniform','distance'],
    'metric':['euclidean','manhattan']
}

In [ ]:
grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid=params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

In [ ]:
grid.fit(X_train_scaled, y_train)

In [ ]:
print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

In [ ]:
best_knn = grid.best_estimator_

y_pred = best_knn.predict(X_test_scaled)

print("Test Accuracy:",accuracy_score(y_test,y_pred))

## DECISION TREE

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train,y_train)

In [ ]:
y_pred_dt = dt.predict(X_test)
y_pred_dt

In [ ]:
accuracy = accuracy_score(y_test,y_pred_dt)
print(f"Accuracy:",accuracy*100,"%")

In [ ]:
params = {

    'criterion':['gini','entropy'],

    'max_depth':[3,5,10,15,None],

    'min_samples_split':[2,5,10],

    'min_samples_leaf':[1,2,4]
}

In [ ]:
grid = GridSearchCV(
    DecisionTreeClassifier(
        random_state=42
    ),
    params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

In [ ]:
grid.fit(X_train,y_train)

In [ ]:
print(grid.best_params_)

In [ ]:
best_dt = grid.best_estimator_

y_pred = best_dt.predict(X_test)

print(f"Accuracy:",
    accuracy_score(y_test,y_pred)*100,"%"
)

## RANDOM FOREST

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train,y_train)

In [ ]:
y_pred_rf = rf.predict(X_test)
y_pred_rf

In [ ]:
accuracy = accuracy_score(y_test,y_pred_rf)
print(f"Accuracy:",accuracy*100,"%")

In [ ]:
params = {

    'n_estimators':[100,200,300],
    'max_depth':[10,20,30,None],
    'min_samples_split':[2,5,10],
    'min_samples_leaf':[1,2,4]
}

In [ ]:
grid = GridSearchCV(
    RandomForestClassifier(
        random_state=42
    ),
    params,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)


In [ ]:
grid.fit(X_train,y_train)

In [ ]:
print(grid.best_params_)

In [ ]:
best_rf = grid.best_estimator_

y_pred = best_rf.predict(X_test)

print(f"Accuracy:",accuracy_score(y_test,y_pred)*100,"%"
)

In [ ]:
results = []

In [ ]:
# KNN
y_pred_knn = best_knn.predict(X_test_scaled)

results.append([
    "KNN",
    accuracy_score(y_test, y_pred_knn)
])

In [ ]:
# Logistic Regression
y_pred_lr = best_lr.predict(X_test_scaled)

results.append([
    "Logistic Regression",
    accuracy_score(y_test, y_pred_lr)
])


In [ ]:
# Decision Tree
y_pred_dt = best_dt.predict(X_test)

results.append([
    "Decision Tree",
    accuracy_score(y_test, y_pred_dt)
])


In [ ]:
# Random Forest
y_pred_rf = best_rf.predict(X_test)

results.append([
    "Random Forest",
    accuracy_score(y_test, y_pred_rf)
])


In [ ]:
# COMPARISION TABLE 
comparison = pd.DataFrame(
    results,
    columns=["Model", "Accuracy"]
)
comparison = comparison.sort_values(
    by="Accuracy",
    ascending=False
)

In [ ]:
print(comparison)

In [ ]:
import joblib 
joblib.dump(RandomForestClassifier,'Shoppers_intention_model.pkl')